In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_SZ = 0.8
VALID_SZ = 1.0 - TRAIN_SZ

# Load the data
data = pd.read_csv('../datasets/melbourne-housing-snapshot/melb_data.csv')

# Select predictors and target
# To keep things simple, we'll use only numerical predictors
y = data.Price
X = data.drop(['Price'], axis=1).select_dtypes(exclude=['object'])

# Divide data into training and validation subsets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                      train_size=TRAIN_SZ, test_size=VALID_SZ,
                                                      random_state=0)

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [3]:
### APPROACH #1: Drop Columns with Missing Values

# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns 
                     if X_train[col].isnull().any()]

drop_X_train = X_train.drop(cols_with_missing, axis=1)
drop_X_valid = X_valid.drop(cols_with_missing, axis=1)

drop_mae = score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)
print(f"MAE from Approach 1 (Drop columns with missing values): {drop_mae}")

MAE from Approach 1 (Drop columns with missing values): 183550.22137772635


In [4]:
### APPROACH #2: Imputation
from sklearn.impute import SimpleImputer

imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(imputer.transform(X_valid))

imputed_mae = score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid)
print(f"MAE from Approach 2 (Imputation): {imputed_mae}")

MAE from Approach 2 (Imputation): 178166.46269899711


In [5]:
### Approach #3: An Extension to Imputation

# Make copy to avoid changing original data (when imputing)
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

# Imputation
imputed_X_train_plus = pd.DataFrame(imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(imputer.transform(X_valid_plus))

# Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

imputed_mae_plus =score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid)
print(f"MAE from Approach 3 (An Extension to Imputation): {imputed_mae_plus}")

MAE from Approach 3 (An Extension to Imputation): 178927.503183954


In [6]:
# Shape of training data (num_rows, num_columns)
print(X_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(10864, 12)
Car               49
BuildingArea    5156
YearBuilt       4307
dtype: int64
